In [ ]:
import numpy as np
import pandas as pd
import os
import json

RAW_PATH = "../data/raw/Electronics_5.json"
# ============================================================
# 1️⃣ LOAD RAW
# ============================================================

def load_json(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

print("Loading raw...")
df = load_json(RAW_PATH)

In [ ]:
SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

OUT_DIR = "rl_dataset_usersplit"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1) Mapear user_id (si ya lo tienes, reutilízalo)
# ------------------------------------------------------------
user2id = {u:i for i,u in enumerate(df["reviewerID"].unique())}
df = df.copy()
df["user_id"] = df["reviewerID"].map(user2id).astype(np.int32)

# ------------------------------------------------------------
# 2) Longitud por usuario
# ------------------------------------------------------------
user_len = df.groupby("user_id").size().rename("user_len").reset_index()

print("Users:", len(user_len))
print("User_len stats:\n", user_len["user_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

# ------------------------------------------------------------
# 3) Crear bins para estratificar (por cuantiles)
#    (si todos los usuarios tienen >=5, esto funciona muy bien)
# ------------------------------------------------------------
# 10 bins por defecto; si fallase por duplicados, reduce a 5
n_bins = 10
try:
    user_len["len_bin"] = pd.qcut(user_len["user_len"], q=n_bins, labels=False, duplicates="drop")
except ValueError:
    n_bins = 5
    user_len["len_bin"] = pd.qcut(user_len["user_len"], q=n_bins, labels=False, duplicates="drop")

print("\nBins usados:", user_len["len_bin"].nunique())
print("Users por bin:\n", user_len["len_bin"].value_counts().sort_index())

# ------------------------------------------------------------
# 4) Split estratificado por bin
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)

train_users = []
val_users = []
test_users = []

for b, g in user_len.groupby("len_bin"):
    users = g["user_id"].to_numpy()
    rng.shuffle(users)

    n = len(users)
    n_train = int(round(n * TRAIN_FRAC))
    n_val = int(round(n * VAL_FRAC))
    # asegurar que suma exacta
    n_test = n - n_train - n_val

    train_users.append(users[:n_train])
    val_users.append(users[n_train:n_train+n_val])
    test_users.append(users[n_train+n_val:])

train_users = np.concatenate(train_users)
val_users = np.concatenate(val_users)
test_users = np.concatenate(test_users)

# sanity checks
train_set = set(train_users.tolist())
val_set = set(val_users.tolist())
test_set = set(test_users.tolist())

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0
assert len(train_set) + len(val_set) + len(test_set) == user_len["user_id"].nunique()

print("\nUser split sizes:")
print("train:", len(train_users), "val:", len(val_users), "test:", len(test_users))
print("ratios:", len(train_users)/len(user_len), len(val_users)/len(user_len), len(test_users)/len(user_len))

# ------------------------------------------------------------
# 5) Asignar split a cada evento
# ------------------------------------------------------------
df["split"] = "train"
df.loc[df["user_id"].isin(val_set), "split"] = "val"
df.loc[df["user_id"].isin(test_set), "split"] = "test"

print("\nEvent split ratios:")
print(df["split"].value_counts(normalize=True))

# comprobar que el split de users mantiene distribución de longitudes parecida
def summarize_len(name, uids):
    lens = user_len[user_len["user_id"].isin(uids)]["user_len"]
    return pd.Series({
        "users": len(lens),
        "mean_len": lens.mean(),
        "median_len": lens.median(),
        "p90_len": np.percentile(lens, 90),
        "p99_len": np.percentile(lens, 99),
        "max_len": lens.max(),
    }, name=name)

print("\nUser length summary by split:")
print(pd.concat([
    summarize_len("train", train_set),
    summarize_len("val", val_set),
    summarize_len("test", test_set),
], axis=1).T)

# ------------------------------------------------------------
# 6) Guardar split de usuarios (reproducibilidad)
# ------------------------------------------------------------
np.savez_compressed(
    os.path.join(OUT_DIR, "user_split_ids.npz"),
    train_users=train_users.astype(np.int32),
    val_users=val_users.astype(np.int32),
    test_users=test_users.astype(np.int32),
    seed=np.int64(SEED),
)

print("\nSaved:", os.path.join(OUT_DIR, "user_split_ids.npz"))